In [45]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "1"

import json
import torch
import torch.nn.functional as F

from PIL import Image
from torchvision import transforms
from transformers import AutoModelForImageClassification
from peft import PeftModel, PeftConfig

# ============================================================
# CONFIG
# ============================================================

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

CLASSIFIER_PATH = "/mnt/extended-home/dzakaaufa/models/dinov2/best_dinov2_lora"
CLASS_MAPPING_PATH = os.path.join(CLASSIFIER_PATH, "class_mapping.json")

# ============================================================
# LOAD CLASS MAPPING
# ============================================================

with open(CLASS_MAPPING_PATH, "r") as f:
    IDX_TO_CLASS = json.load(f)

# ============================================================
# IMAGE TRANSFORM
# ============================================================

dinov2_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

# ============================================================
# LOAD DINOv2 + LoRA
# ============================================================

def load_classification_model(lora_path):
    config = PeftConfig.from_pretrained(lora_path)

    base_model = AutoModelForImageClassification.from_pretrained(
        config.base_model_name_or_path,
        num_labels=len(IDX_TO_CLASS),
        ignore_mismatched_sizes=True
    )

    model = PeftModel.from_pretrained(
        base_model,
        lora_path
    )

    model.to(DEVICE)
    model.eval()

    print("✓ DINOv2 Classifier berhasil dimuat")
    return model

# ============================================================
# CLASSIFIER PIPELINE
# ============================================================

class BatikClassifier:

    def __init__(self, classifier_path):
        self.model = load_classification_model(classifier_path)

    def predict(self, image_path):

        image = Image.open(image_path).convert("RGB")

        img_tensor = (
            dinov2_transform(image)
            .unsqueeze(0)
            .to(DEVICE)
        )

        with torch.no_grad():
            outputs = self.model(img_tensor)

            probs = F.softmax(outputs.logits, dim=1)[0]

            confidence, class_idx = torch.max(probs, dim=0)

            predicted_class = IDX_TO_CLASS[str(class_idx.item())]

            top5_probs, top5_indices = torch.topk(probs, k=5)

        for prob, idx in zip(top5_probs, top5_indices):
            label = IDX_TO_CLASS[str(idx.item())]
            print(f"{label:<25} {prob.item():.2%}")

        return {
            "file": image_path,
            "predicted_class": predicted_class,
            "confidence": confidence.item()
        }

# ============================================================
# MAIN
# ============================================================

if __name__ == "__main__":

    classifier = BatikClassifier(CLASSIFIER_PATH)

    test_img = "/mnt/extended-home/dzakaaufa/dataset/test/lamongan.jpg"

    result = classifier.predict(test_img)

    print("\n" + "="*50)
    print("HASIL KLASIFIKASI DINOv2")
    print("="*50)

    print(f"File       : {result['file']}")
    print(f"Kelas      : {result['predicted_class']}")
    print(f"Confidence : {result['confidence']:.2%}")

    print("="*50)

Some weights of Dinov2ForImageClassification were not initialized from the model checkpoint at facebook/dinov2-large and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


✓ DINOv2 Classifier berhasil dimuat
malang                    89.43%
lamongan                  5.38%
betawi                    2.19%
trenggalek                1.69%
truntum                   0.27%

HASIL KLASIFIKASI DINOv2
File       : /mnt/extended-home/dzakaaufa/dataset/test/lamongan.jpg
Kelas      : malang
Confidence : 89.43%


In [4]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import json
import torch
import torch.nn.functional as F
import pandas as pd
from tqdm import tqdm
import numpy as np
import random

from PIL import Image
from torchvision import transforms
from torch.utils.data import Dataset, DataLoader
from transformers import AutoModelForImageClassification
from peft import PeftModel, PeftConfig

# ── Patch torch < 2.4 ─────────────────────────────────────────────
if not hasattr(torch.nn.Module, "set_submodule"):
    def set_submodule(self, target, module):
        atoms = target.split(".")
        name = atoms.pop(-1)
        mod = self
        for item in atoms:
            mod = getattr(mod, item)
        setattr(mod, name, module)
    torch.nn.Module.set_submodule = set_submodule

# ============================================================
# 1. SETUP & CONFIG
# ============================================================

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

CLASSIFIER_PATH = "/mnt/extended-home/dzakaaufa/leakage/models150/dino"
CLASS_MAPPING_PATH = os.path.join(CLASSIFIER_PATH, "class_mapping.json")
OUTPUT_CSV = "/mnt/extended-home/dzakaaufa/skripsi/full/classification_only.csv"

IMAGE_FOLDER = "/mnt/extended-home/dzakaaufa/dataset/all_images_captioning"
GROUND_TRUTH_FILE = "/mnt/extended-home/dzakaaufa/leakage/models150/data_final_150_150.csv"
BATCH_SIZE = 32

# Load class mapping
with open(CLASS_MAPPING_PATH, "r") as f:
    IDX_TO_CLASS = json.load(f)

# Buat mapping sebaliknya untuk pencarian indeks Ground Truth
CLASS_TO_IDX = {v: int(k) for k, v in IDX_TO_CLASS.items()}

# Image Transform
dinov2_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# ============================================================
# 2. DATASET & UTILITIES
# ============================================================

class BatikEvalDataset(Dataset):
    def __init__(self, image_paths, transform=None):
        self.image_paths = image_paths
        self.transform = transform

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        image = Image.open(img_path).convert("RGB")
        if self.transform:
            image = self.transform(image)
        return image, img_path

def normalize_filename(path):
    return os.path.basename(str(path)).strip().lower()

def get_ground_truth_mapping(gt_file_path):
    class_mapping = {}
    test_images_only = set()

    print(f"-> Memuat ground truth master dari {gt_file_path}...")
    gt_df = pd.read_csv(gt_file_path)

    if 'split' in gt_df.columns:
        gt_df = gt_df[gt_df['split'] == 'test']
        print(f"-> Berhasil menyaring data. Ditemukan {len(gt_df)} baris split 'test'.")
    else:
        print("[WARN] Kolom 'split' tidak ditemukan! Memproses semua baris.")

    for _, row in gt_df.iterrows():
        filename = normalize_filename(row['image_path'])
        class_mapping[filename] = str(row['class'])
        test_images_only.add(filename)

    return class_mapping, test_images_only

# ============================================================
# 3. LOAD CLASSIFIER MODEL
# ============================================================

def load_classification_model(lora_path):
    print(f"-> Memuat DINOv2 Classifier LoRA dari: {lora_path}")
    config = PeftConfig.from_pretrained(lora_path)
    base_model = AutoModelForImageClassification.from_pretrained(
        config.base_model_name_or_path,
        num_labels=len(IDX_TO_CLASS),
        ignore_mismatched_sizes=True
    )
    model = PeftModel.from_pretrained(base_model, lora_path)
    model.to(DEVICE)
    model.eval()
    print("✓ Model berhasil dimuat.")
    return model
def normalize_label(label):
    """Menyeragamkan format teks agar pencocokan Akurasi valid"""
    return str(label).lower().replace("_", " ").strip()

if __name__ == "__main__":
    
    # 1. Load Ground Truth
    gt_class_mapping, test_images_whitelist = get_ground_truth_mapping(GROUND_TRUTH_FILE)

    # 2. Filter list gambar yang ada di folder sesuai whitelist 'test'
    valid_extensions = (".png", ".jpg", ".jpeg", ".webp")
    image_paths = [
        os.path.join(IMAGE_FOLDER, f) for f in os.listdir(IMAGE_FOLDER)
        if f.lower().endswith(valid_extensions) and normalize_filename(f) in test_images_whitelist
    ]
    print(f"[INFO] Total {len(image_paths)} gambar siap diuji.")

    # 3. Load Model DINOv2 murni (Tanpa Qwen)
    model = load_classification_model(CLASSIFIER_PATH)

    # 4. Buat Dataset & DataLoader untuk Batch Inference super cepat
    dataset = BatikEvalDataset(image_paths, transform=dinov2_transform)
    dataloader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=4)

    extracted_data = []

    print("\n[*] Menjalankan Batch Inference...")
    with torch.no_grad():
        for imgs, paths in tqdm(dataloader, desc="Evaluasi Kelas"):
            imgs = imgs.to(DEVICE)
            outputs = model(imgs)
            probs_batch = F.softmax(outputs.logits, dim=1)  # (Batch_Size, Num_Classes)

            # Ambil prediksi nilai tertinggi (Top-1)
            confidences, class_indices = torch.max(probs_batch, dim=1)

            for i, (path, conf, class_idx) in enumerate(zip(paths, confidences, class_indices)):
                filename = normalize_filename(path)
                
                # --- NORMALISASI LABEL DI SINI ---
                gt_class_raw = gt_class_mapping.get(filename, "UNKNOWN")
                predicted_class_raw = IDX_TO_CLASS[str(class_idx.item())]
                
                gt_class_norm = normalize_label(gt_class_raw)
                predicted_class_norm = normalize_label(predicted_class_raw)

                # Evaluasi Benar/Salah setelah teks diseragamkan
                is_correct = int(predicted_class_norm == gt_class_norm)

                # --- Perhitungan Reciprocal Rank (untuk MMR) ---
                # Mengurutkan probabilitas dari yang tertinggi ke terendah
                sorted_probs, sorted_indices = torch.sort(probs_batch[i], descending=True)
                
                # Normalisasi seluruh nama kelas dalam daftar urutan
                sorted_classes_norm = [normalize_label(IDX_TO_CLASS[str(idx.item())]) for idx in sorted_indices]
                
                # Cari posisi Ground Truth di dalam urutan probabilitas tersebut (Rank ke-berapa)
                if gt_class_norm in sorted_classes_norm:
                    rank = sorted_classes_norm.index(gt_class_norm) + 1  # 1-based index
                    reciprocal_rank = 1.0 / rank
                else:
                    rank = len(sorted_classes_norm) + 1
                    reciprocal_rank = 0.0

                extracted_data.append({
                    "image_path": path,
                    "ground_truth_class": gt_class_raw, # Tetap simpan format asli di CSV
                    "predicted_class": predicted_class_raw, # Tetap simpan format asli di CSV
                    "confidence": conf.item(),
                    "is_correct": is_correct,
                    "predicted_rank_of_gt": rank,
                    "reciprocal_rank": reciprocal_rank
                })

    # 5. Save & Tampilkan Ringkasan Metrik
    if extracted_data:
        df_results = pd.DataFrame(extracted_data)
        os.makedirs(os.path.dirname(OUTPUT_CSV), exist_ok=True)
        df_results.to_csv(OUTPUT_CSV, index=False)
        print(f"\n[SUKSES] Hasil evaluasi disimpan ke → {OUTPUT_CSV}")

        # Perhitungan Metrik Akhir
        accuracy = df_results["is_correct"].mean()
        mrr = df_results["reciprocal_rank"].mean()
        mean_confidence = df_results["confidence"].mean()
        mean_conf_correct = df_results[df_results["is_correct"] == 1]["confidence"].mean()
        mean_conf_incorrect = df_results[df_results["is_correct"] == 0]["confidence"].mean()

        print("\n" + "="*50)
        print("RINGKASAN METRIK KLASIFIKASI DINOv2")
        print("="*50)
        print(f"Total Citra Diuji        : {len(df_results)}")
        print(f"Akurasi Klasifikasi      : {accuracy:.2%}")
        print(f"Mean Reciprocal Rank(MRR): {mrr:.4f}")
        print(f"Rata-rata Confidence     : {mean_confidence:.2%}")
        print(f"Rata-rata Conf (Benar)   : {mean_conf_correct:.2%}")
        print(f"Rata-rata Conf (Salah)   : {mean_conf_incorrect:.2%}")
        print("="*50)

-> Memuat ground truth master dari /mnt/extended-home/dzakaaufa/leakage/models150/data_final_150_150.csv...
-> Berhasil menyaring data. Ditemukan 150 baris split 'test'.
[INFO] Total 150 gambar siap diuji.
-> Memuat DINOv2 Classifier LoRA dari: /mnt/extended-home/dzakaaufa/leakage/models150/dino


Some weights of Dinov2ForImageClassification were not initialized from the model checkpoint at facebook/dinov2-large and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


✓ Model berhasil dimuat.

[*] Menjalankan Batch Inference...


Evaluasi Kelas: 100%|██████████| 5/5 [00:22<00:00,  4.58s/it]


[SUKSES] Hasil evaluasi disimpan ke → /mnt/extended-home/dzakaaufa/skripsi/full/classification_only.csv

RINGKASAN METRIK KLASIFIKASI DINOv2
Total Citra Diuji        : 150
Akurasi Klasifikasi      : 64.00%
Mean Reciprocal Rank(MRR): 0.7780
Rata-rata Confidence     : 66.83%
Rata-rata Conf (Benar)   : 71.66%
Rata-rata Conf (Salah)   : 58.26%


In [3]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import json
import torch
import torch.nn.functional as F
import pandas as pd
from tqdm import tqdm
import numpy as np
import random

from PIL import Image
from torchvision import transforms
from torch.utils.data import Dataset, DataLoader
from transformers import AutoModelForImageClassification

# ── Patch torch < 2.4 ─────────────────────────────────────────────
if not hasattr(torch.nn.Module, "set_submodule"):
    def set_submodule(self, target, module):
        atoms = target.split(".")
        name = atoms.pop(-1)
        mod = self
        for item in atoms:
            mod = getattr(mod, item)
        setattr(mod, name, module)
    torch.nn.Module.set_submodule = set_submodule

# ============================================================
# 1. SETUP & CONFIG
# ============================================================

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

# Pastikan path ini menunjuk ke folder penyimpanan model Full Fine-Tuning 
# (misal: "best_dinov2_full_ft" dari script training sebelumnya)
CLASSIFIER_PATH = "/mnt/extended-home/dzakaaufa/skripsi/models/dinov2_full"
CLASS_MAPPING_PATH = os.path.join(CLASSIFIER_PATH, "class_mapping.json")
OUTPUT_CSV = "/mnt/extended-home/dzakaaufa/skripsi/full/classification_only.csv"

IMAGE_FOLDER = "/mnt/extended-home/dzakaaufa/dataset/all_images_captioning"
GROUND_TRUTH_FILE = "/mnt/extended-home/dzakaaufa/leakage/models150/data_final_150_150.csv"
BATCH_SIZE = 32

# Load class mapping
with open(CLASS_MAPPING_PATH, "r") as f:
    IDX_TO_CLASS = json.load(f)

# Buat mapping sebaliknya untuk pencarian indeks Ground Truth
CLASS_TO_IDX = {v: int(k) for k, v in IDX_TO_CLASS.items()}

# Image Transform
dinov2_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# ============================================================
# 2. DATASET & UTILITIES
# ============================================================

class BatikEvalDataset(Dataset):
    def __init__(self, image_paths, transform=None):
        self.image_paths = image_paths
        self.transform = transform

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        image = Image.open(img_path).convert("RGB")
        if self.transform:
            image = self.transform(image)
        return image, img_path

def normalize_filename(path):
    return os.path.basename(str(path)).strip().lower()

def get_ground_truth_mapping(gt_file_path):
    class_mapping = {}
    test_images_only = set()

    print(f"-> Memuat ground truth master dari {gt_file_path}...")
    gt_df = pd.read_csv(gt_file_path)

    if 'split' in gt_df.columns:
        gt_df = gt_df[gt_df['split'] == 'test']
        print(f"-> Berhasil menyaring data. Ditemukan {len(gt_df)} baris split 'test'.")
    else:
        print("[WARN] Kolom 'split' tidak ditemukan! Memproses semua baris.")

    for _, row in gt_df.iterrows():
        filename = normalize_filename(row['image_path'])
        class_mapping[filename] = str(row['class'])
        test_images_only.add(filename)

    return class_mapping, test_images_only

# ============================================================
# 3. LOAD CLASSIFIER MODEL (FULL FINE-TUNING)
# ============================================================

def load_classification_model(model_path):
    print(f"-> Memuat DINOv2 Classifier (Full Fine-Tuned) dari: {model_path}")
    
    # Langsung memuat model utuh tanpa PEFT/LoRA
    model = AutoModelForImageClassification.from_pretrained(
        model_path,
        num_labels=len(IDX_TO_CLASS),
        ignore_mismatched_sizes=True
    )
    
    model.to(DEVICE)
    model.eval()
    print("✓ Model berhasil dimuat.")
    return model

def normalize_label(label):
    """Menyeragamkan format teks agar pencocokan Akurasi valid"""
    return str(label).lower().replace("_", " ").strip()

if __name__ == "__main__":
    
    # 1. Load Ground Truth
    gt_class_mapping, test_images_whitelist = get_ground_truth_mapping(GROUND_TRUTH_FILE)

    # 2. Filter list gambar yang ada di folder sesuai whitelist 'test'
    valid_extensions = (".png", ".jpg", ".jpeg", ".webp")
    image_paths = [
        os.path.join(IMAGE_FOLDER, f) for f in os.listdir(IMAGE_FOLDER)
        if f.lower().endswith(valid_extensions) and normalize_filename(f) in test_images_whitelist
    ]
    print(f"[INFO] Total {len(image_paths)} gambar siap diuji.")

    # 3. Load Model DINOv2 Full Fine-Tuned
    model = load_classification_model(CLASSIFIER_PATH)

    # 4. Buat Dataset & DataLoader untuk Batch Inference super cepat
    dataset = BatikEvalDataset(image_paths, transform=dinov2_transform)
    dataloader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=4)

    extracted_data = []

    print("\n[*] Menjalankan Batch Inference...")
    with torch.no_grad():
        for imgs, paths in tqdm(dataloader, desc="Evaluasi Kelas"):
            imgs = imgs.to(DEVICE)
            outputs = model(imgs)
            probs_batch = F.softmax(outputs.logits, dim=1)  # (Batch_Size, Num_Classes)

            # Ambil prediksi nilai tertinggi (Top-1)
            confidences, class_indices = torch.max(probs_batch, dim=1)

            for i, (path, conf, class_idx) in enumerate(zip(paths, confidences, class_indices)):
                filename = normalize_filename(path)
                
                # --- NORMALISASI LABEL DI SINI ---
                gt_class_raw = gt_class_mapping.get(filename, "UNKNOWN")
                predicted_class_raw = IDX_TO_CLASS[str(class_idx.item())]
                
                gt_class_norm = normalize_label(gt_class_raw)
                predicted_class_norm = normalize_label(predicted_class_raw)

                # Evaluasi Benar/Salah setelah teks diseragamkan
                is_correct = int(predicted_class_norm == gt_class_norm)

                # --- Perhitungan Reciprocal Rank (untuk MRR) ---
                # Mengurutkan probabilitas dari yang tertinggi ke terendah
                sorted_probs, sorted_indices = torch.sort(probs_batch[i], descending=True)
                
                # Normalisasi seluruh nama kelas dalam daftar urutan
                sorted_classes_norm = [normalize_label(IDX_TO_CLASS[str(idx.item())]) for idx in sorted_indices]
                
                # Cari posisi Ground Truth di dalam urutan probabilitas tersebut (Rank ke-berapa)
                if gt_class_norm in sorted_classes_norm:
                    rank = sorted_classes_norm.index(gt_class_norm) + 1  # 1-based index
                    reciprocal_rank = 1.0 / rank
                else:
                    rank = len(sorted_classes_norm) + 1
                    reciprocal_rank = 0.0

                extracted_data.append({
                    "image_path": path,
                    "ground_truth_class": gt_class_raw, # Tetap simpan format asli di CSV
                    "predicted_class": predicted_class_raw, # Tetap simpan format asli di CSV
                    "confidence": conf.item(),
                    "is_correct": is_correct,
                    "predicted_rank_of_gt": rank,
                    "reciprocal_rank": reciprocal_rank
                })

    # 5. Save & Tampilkan Ringkasan Metrik
    if extracted_data:
        df_results = pd.DataFrame(extracted_data)
        os.makedirs(os.path.dirname(OUTPUT_CSV), exist_ok=True)
        df_results.to_csv(OUTPUT_CSV, index=False)
        print(f"\n[SUKSES] Hasil evaluasi disimpan ke → {OUTPUT_CSV}")

        # Perhitungan Metrik Akhir
        accuracy = df_results["is_correct"].mean()
        mrr = df_results["reciprocal_rank"].mean()
        mean_confidence = df_results["confidence"].mean()
        mean_conf_correct = df_results[df_results["is_correct"] == 1]["confidence"].mean()
        mean_conf_incorrect = df_results[df_results["is_correct"] == 0]["confidence"].mean()

        print("\n" + "="*50)
        print("RINGKASAN METRIK KLASIFIKASI DINOv2 (FULL FT)")
        print("="*50)
        print(f"Total Citra Diuji        : {len(df_results)}")
        print(f"Akurasi Klasifikasi      : {accuracy:.2%}")
        print(f"Mean Reciprocal Rank(MRR): {mrr:.4f}")
        print(f"Rata-rata Confidence     : {mean_confidence:.2%}")
        print(f"Rata-rata Conf (Benar)   : {mean_conf_correct:.2%}")
        print(f"Rata-rata Conf (Salah)   : {mean_conf_incorrect:.2%}")
        print("="*50)

-> Memuat ground truth master dari /mnt/extended-home/dzakaaufa/leakage/models150/data_final_150_150.csv...
-> Berhasil menyaring data. Ditemukan 150 baris split 'test'.
[INFO] Total 150 gambar siap diuji.
-> Memuat DINOv2 Classifier (Full Fine-Tuned) dari: /mnt/extended-home/dzakaaufa/skripsi/models/dinov2_full
✓ Model berhasil dimuat.

[*] Menjalankan Batch Inference...


Evaluasi Kelas: 100%|██████████| 5/5 [00:24<00:00,  4.92s/it]


[SUKSES] Hasil evaluasi disimpan ke → /mnt/extended-home/dzakaaufa/skripsi/full/classification_only.csv

RINGKASAN METRIK KLASIFIKASI DINOv2 (FULL FT)
Total Citra Diuji        : 150
Akurasi Klasifikasi      : 51.33%
Mean Reciprocal Rank(MRR): 0.6985
Rata-rata Confidence     : 67.66%
Rata-rata Conf (Benar)   : 74.01%
Rata-rata Conf (Salah)   : 60.95%
